# Linha Categórico — Casos

Para as três variáveis categóricas de Casos (Faixa Etária, Escolaridade
e Raça/Cor), um gráfico de linhas mostra a **Quantidade** (valor
absoluto) por categoria ao longo dos anos -- uma linha por categoria,
todas no mesmo eixo Y, sem taxa e sem segundo eixo (ver decisão abaixo).

Cada categoria é distinguível por duas camadas independentes -- cor
(paleta forte e qualitativa, `PALETA_CATEGORIAS_FORTE` em
`configuracao_layout_graficos.py`) e símbolo do marcador
(`MARCADORES_CATEGORIAS`) -- para continuar legível também em impressão
preto e branco. O marcador é maior que o padrão dos demais gráficos de
linha, com borda branca fina, e o valor de cada ponto é rotulado acima
do marcador (1 a cada `FREQUENCIA_ROTULO_LINHA_CATEGORICA` anos, para
não saturar a figura com décadas de pontos em até 8 linhas).

QUANTIDADE estadual já vem pronta do CSV agregado por ano
(`tabela_regional_agregada_por_ano.csv`, gerado por
`calculo_taxa.ipynb`) -- nenhuma agregação é recalculada aqui, só
recortada (`preparar_categorica_estadual_por_ano`) e pivotada para o
gráfico.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

from caminho_utilitarios import preparar_sys_path
preparar_sys_path()


PosixPath('/home/anna/tcc/Seleção e Transformação/População')

In [2]:
from utilitarios_comuns import (
    PASTA_POPULACAO,
    ESPECIFICACOES_VARIAVEIS_CATEGORICAS,
    SUFIXO_VARIAVEL_ESTADUAL,
    NOME_ARQUIVO_AGREGADO_CATEGORICO_POR_ANO,
    registrar_log,
)
from utilitarios_graficos_populacionais import (
    carregar_taxa_por_regional,
    preparar_categorica_estadual_por_ano,
    ordenar_categorias,
    gerar_paleta_categorica_forte,
    gerar_marcadores_categoria,
    gerar_posicoes_rotulo_categoria,
    montar_grafico_linha_categorica_absoluto,
    remover_anos_sem_dado,
    salvar_figura,
)


### Configuração

In [3]:
PASTA_TAXAS = PASTA_POPULACAO / "Taxa"
PASTA_SAIDA_GRAFICOS = Path.cwd() / "Gráficos" / "Linha Categórico"

SALVAR_FIGURAS = False

ANO_INICIAL_EXIBIDO = None
ANO_FINAL_EXIBIDO = None

LARGURA_FIGURA = 1100
ALTURA_FIGURA = 950

# Espaçamento do eixo Y (dtick) por variável; as ausentes usam o automático do Plotly
DTICK_EIXO_Y_POR_VARIAVEL = {
    "Casos por Faixa Etária": 400,
    "Casos por Escolaridade": 400,
}

LISTA_VARIAVEIS_CATEGORICAS = [
    {
        "NOME": nome,
        "SUFIXO": SUFIXO_VARIAVEL_ESTADUAL[nome],
        "ANO_INICIAL": ANO_INICIAL_EXIBIDO or especificacao["ANO_INICIAL"],
        "ANO_FINAL": ANO_FINAL_EXIBIDO or especificacao["ANO_FINAL"],
        "DTICK_EIXO_Y": DTICK_EIXO_Y_POR_VARIAVEL.get(nome),
    }
    for nome, especificacao in ESPECIFICACOES_VARIAVEIS_CATEGORICAS.items()
]

tabela_agregada_estadual = carregar_taxa_por_regional(PASTA_TAXAS / NOME_ARQUIVO_AGREGADO_CATEGORICO_POR_ANO)


### Gráfico — Quantidade (linha) por Categoria


In [4]:
for configuracao in LISTA_VARIAVEIS_CATEGORICAS:
    nome_da_variavel = configuracao["NOME"]
    registrar_log(f"===== INÍCIO: {nome_da_variavel.upper()} =====")

    _tabela_pivotada_taxa, tabela_pivotada_quantidade = preparar_categorica_estadual_por_ano(
        tabela_agregada_estadual, configuracao["SUFIXO"],
        configuracao["ANO_INICIAL"], configuracao["ANO_FINAL"],
    )

    # Remove os anos sem dado em nenhuma categoria e só depois preenche as ausências com 0
    (tabela_pivotada_quantidade,) = remover_anos_sem_dado(tabela_pivotada_quantidade)
    tabela_pivotada_quantidade = tabela_pivotada_quantidade.fillna(0)

    categorias_ordenadas = ordenar_categorias(tabela_pivotada_quantidade.columns.tolist())
    paleta_de_cores = gerar_paleta_categorica_forte(categorias_ordenadas)
    marcadores_de_categoria = gerar_marcadores_categoria(categorias_ordenadas)
    posicoes_de_rotulo = gerar_posicoes_rotulo_categoria(categorias_ordenadas)

    figura_linha = montar_grafico_linha_categorica_absoluto(
        eixo_x=tabela_pivotada_quantidade.index,
        tabela_pivotada_quantidade=tabela_pivotada_quantidade,
        titulo="",
        paleta_cores=paleta_de_cores, marcadores_categorias=marcadores_de_categoria,
        posicoes_rotulo=posicoes_de_rotulo, dtick_eixo_y=configuracao["DTICK_EIXO_Y"],
        titulo_eixo_x="Ano", ordem_categorias=categorias_ordenadas,
        largura=LARGURA_FIGURA, altura=ALTURA_FIGURA, tickangle_eixo_x=-45, eixo_x_ano_continuo=True,
    )
    figura_linha.show()

    if SALVAR_FIGURAS:
        nome_arquivo_base = nome_da_variavel.lower().replace(" ", "_")
        salvar_figura(figura_linha, f"{nome_arquivo_base}_linha_categorica", PASTA_SAIDA_GRAFICOS)

    registrar_log(f"===== FIM: {nome_da_variavel.upper()} =====\n")


[LOG] ===== INÍCIO: CASOS POR ESCOLARIDADE =====


[LOG] ===== FIM: CASOS POR ESCOLARIDADE =====

[LOG] ===== INÍCIO: CASOS POR RAÇA E COR =====


[LOG] ===== FIM: CASOS POR RAÇA E COR =====

[LOG] ===== INÍCIO: CASOS POR FAIXA ETÁRIA =====


[LOG] ===== FIM: CASOS POR FAIXA ETÁRIA =====

